In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/bahaa1515/EECE-693-project.git"
COLAB_REPO_DIR = Path("/content/EECE-693-project")

def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "src" / "config.py").exists():
            return candidate
    if COLAB_REPO_DIR.exists() and (COLAB_REPO_DIR / "src" / "config.py").exists():
        return COLAB_REPO_DIR
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError as exc:
        raise FileNotFoundError(
            "Could not find the project root. Run this notebook from the repo root, "
            "from the notebooks folder, or clone the repo in Colab first."
        ) from exc
    subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    return COLAB_REPO_DIR

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

import pandas as pd
import numpy as np

from src.config import SMARTWATCH_FILES, DATA_INTERIM, OUTPUT_TABLES


In [40]:
usecols = ["user_key", "date", "time", "activity_type", "intensity", "steps", "hr"]

dtype_map = {
    "user_key": "int32",
    "date": "int16",
    "activity_type": "float32",
    "intensity": "float32",
    "steps": "float32",
    "hr": "float32",
}

smartwatch = pd.concat(
    [pd.read_csv(f, usecols=usecols, dtype=dtype_map) for f in SMARTWATCH_FILES],
    ignore_index=True
)

print(smartwatch.shape)
display(smartwatch.head())

(2101829, 7)


,user_key,date,time,activity_type,intensity,steps,hr
0,113,0,09:35:00,80.0,31.0,0.0,NaN
1,113,0,09:36:00,96.0,6.0,0.0,72.0
2,113,0,09:37:00,80.0,0.0,0.0,73.0
3,113,0,09:38:00,80.0,13.0,0.0,74.0
4,113,0,09:39:00,90.0,20.0,0.0,75.0


In [41]:
before = len(smartwatch)
smartwatch = smartwatch.drop_duplicates()
after = len(smartwatch)

print("Dropped duplicates:", before - after)

Dropped duplicates: 0


In [42]:
before = len(smartwatch)
smartwatch = smartwatch[smartwatch["date"] >= 0].copy()
after = len(smartwatch)

print("Dropped negative-date rows:", before - after)

Dropped negative-date rows: 70


In [43]:
smartwatch.loc[smartwatch["steps"] < 0, "steps"] = np.nan
smartwatch.loc[smartwatch["hr"] <= 0, "hr"] = np.nan

In [44]:
smartwatch["time"] = pd.to_datetime(
    smartwatch["time"], format="%H:%M:%S", errors="coerce"
).dt.time

smartwatch["minutes_from_midnight"] = (
    pd.to_datetime(smartwatch["time"].astype(str), format="%H:%M:%S", errors="coerce").dt.hour * 60
    + pd.to_datetime(smartwatch["time"].astype(str), format="%H:%M:%S", errors="coerce").dt.minute
)

smartwatch["date"] = smartwatch["date"].astype("int64")
smartwatch["minutes_from_midnight"] = smartwatch["minutes_from_midnight"].astype("int64")

smartwatch["relative_minute"] = smartwatch["date"] * 1440 + smartwatch["minutes_from_midnight"]

print(smartwatch[["user_key", "date", "time", "minutes_from_midnight", "relative_minute"]].head(10))
print("date dtype:", smartwatch["date"].dtype)
print("minutes_from_midnight dtype:", smartwatch["minutes_from_midnight"].dtype)
print("relative_minute dtype:", smartwatch["relative_minute"].dtype)
print("Min date:", smartwatch["date"].min())
print("Min relative_minute:", smartwatch["relative_minute"].min())

smartwatch = smartwatch.sort_values(["user_key", "relative_minute"]).reset_index(drop=True)
display(smartwatch.tail())

   user_key  date      time  minutes_from_midnight  relative_minute
0       113     0  09:35:00                    575              575
1       113     0  09:36:00                    576              576
2       113     0  09:37:00                    577              577
3       113     0  09:38:00                    578              578
4       113     0  09:39:00                    579              579
5       113     0  09:40:00                    580              580
6       113     0  09:41:00                    581              581
7       113     0  09:42:00                    582              582
8       113     0  09:43:00                    583              583
9       113     0  09:44:00                    584              584
date dtype: int64
minutes_from_midnight dtype: int64
relative_minute dtype: int64
Min date: 0
Min relative_minute: 0


,user_key,date,time,activity_type,intensity,steps,hr,minutes_from_midnight,relative_minute
2101754,939,178,06:02:00,1.0,76.0,24.0,86.0,362,256682
2101755,939,178,06:03:00,1.0,91.0,38.0,86.0,363,256683
2101756,939,178,06:04:00,1.0,106.0,34.0,86.0,364,256684
2101757,939,178,06:05:00,16.0,108.0,10.0,86.0,365,256685
2101758,939,178,06:06:00,96.0,66.0,0.0,85.0,366,256686


In [45]:
clean_path = DATA_INTERIM / "smartwatch_cleaned.parquet"
smartwatch.to_parquet(clean_path, index=False)

print("Saved to:", clean_path)
print("shape:", smartwatch.shape)

Saved to: C:\Users\user\Desktop\693\project phase 1\data\interim\smartwatch_cleaned.parquet
shape: (2101759, 9)


In [46]:
cleaning_log = pd.DataFrame({
    "metric": [
        "rows_after_cleaning",
        "unique_users",
        "missing_hr_pct",
        "missing_steps_pct"
    ],
    "value": [
        len(smartwatch),
        smartwatch["user_key"].nunique(),
        round(smartwatch["hr"].isna().mean() * 100, 2),
        round(smartwatch["steps"].isna().mean() * 100, 2)
    ]
})

cleaning_log.to_csv(OUTPUT_TABLES / "smartwatch_cleaning_log.csv", index=False)
display(cleaning_log)

,metric,value
0,rows_after_cleaning,2101759.0
1,unique_users,20.0
2,missing_hr_pct,25.3
3,missing_steps_pct,0.0
